In [ ]:
import pandas as pd
import numpy as np
import re
from functools import reduce
import seaborn as sns
import missingno as msno
%matplotlib inline

import matplotlib.pyplot as plt
from scipy.stats import norm

# import data

In [ ]:
data_oa = pd.read_pickle('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/wos_data_and_open_access_upw_with_doi_2002_2021_with_oa_status.pkl')
data_oa

In [ ]:
data_oa['publisher_city']=data_oa['Publisher City'].str.replace('\d+', '').str.strip()

In [ ]:
data_oa.columns

In [ ]:
data_countries = pd.read_pickle('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/wos_data_countries_with_doi_2002_2021_with_oa_status.pkl')
data_countries

In [ ]:
data_countries.columns.to_list()

In [ ]:
data_oa['publication_year'].value_counts()

# 6. relation between funding and OA ?

In [ ]:
data_oa.columns

## group by 'is_oa' 

In [ ]:
data_oa_funding = data_oa[['is_oa','publication_year','funding_yes']].set_index(['is_oa','publication_year'])#
data_oa_funding

In [ ]:
#data_oa_funding.groupby(['is_oa','funding_yes']).count()

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_funding[col]) for col in data_oa_funding]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

In [ ]:
#dfs['publication_year'].value_counts()

In [ ]:
data_oa_grp_funding = dfs.groupby(['publication_year','is_oa']).count().T
data_oa_grp_funding['total', 'total'] = data_oa_grp_funding.T.sum()
data_oa_grp_funding

In [ ]:
funding_year = dfs.groupby(['publication_year']).count().T.iloc[1:]
funding_year['total'] = funding_year.T.sum()
funding_year

In [ ]:
for col in data_oa_grp_funding.columns:
    for row in data_oa_grp_funding.index.to_list():
        data_oa_grp_funding[col] = data_oa_grp_funding[col]
        data_oa_grp_funding[col[0],col[1]+'_%'] = data_oa_grp_funding[col]/funding_year[col[0]]*100
        data_oa_grp_funding[col[0],col[1]+'_%_z_score'] = (data_oa_grp_funding[col[0],col[1]+'_%'] - data_oa_grp_funding[col[0],col[1]+'_%'].mean()) /data_oa_grp_funding[col[0],col[1]+'_%'].std()

#save data
data_oa_grp_funding.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_grp_year_funding_z_score.csv')
data_oa_grp_funding

In [ ]:
data_oa_grp_funding.columns[0][0]

In [ ]:
# for col in data_oa_grp_funding.columns:
#     for row in data_oa_grp_funding.index.to_list():
#         data_oa_grp_funding[col] = data_oa_grp_funding[col]
#         data_oa_grp_funding[col[0],col[1]+'_%'] = data_oa_grp_funding[col]/data_oa_grp_funding['total','total']*100
#         data_oa_grp_funding[col[0],col[1]+'_%_z_score'] = (data_oa_grp_funding[col[0],col[1]+'_%'] - data_oa_grp_funding[col[0],col[1]+'_%'].mean()) /data_oa_grp_funding[col[0],col[1]+'_%'].std()

# #save data
# data_oa_grp_funding.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_grp_year_funding_z_score.csv')
# data_oa_grp_funding

## group by 'oa_status' 

In [ ]:
data_oa_status_funding = data_oa[['publication_year','oa_status','funding_yes']].set_index(['oa_status','publication_year'])
data_oa_status_funding

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_status_funding[col]) for col in data_oa_status_funding]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

In [ ]:
data_oa_status_grp_funding = dfs.groupby(['publication_year','oa_status']).count().T
data_oa_status_grp_funding['total', 'total'] = data_oa_status_grp_funding.T.sum()
data_oa_status_grp_funding

In [ ]:
for col in data_oa_status_grp_funding.columns:
    for row in data_oa_status_grp_funding.index.to_list():
        data_oa_status_grp_funding[col] = data_oa_status_grp_funding[col]
        data_oa_status_grp_funding[col[0],col[1]+'_%'] = data_oa_status_grp_funding[col]/funding_year[col[0]]*100
        data_oa_status_grp_funding[col[0],col[1]+'_%_z_score'] = (data_oa_status_grp_funding[col[0],col[1]+'_%'] - data_oa_status_grp_funding[col[0],col[1]+'_%'].mean()) /data_oa_status_grp_funding[col[0],col[1]+'_%'].std()

#save data
data_oa_status_grp_funding.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_status_grp_year_funding_z_score.csv')
data_oa_status_grp_funding

# 7. relation between document types and OA ?

In [ ]:
data_oa['genre'].value_counts()

## group by 'is_oa' 

In [ ]:
data_oa_doc_types = data_oa[['is_oa','publication_year','genre']].set_index(['is_oa','publication_year'])
data_oa_doc_types

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_doc_types[col]) for col in data_oa_doc_types]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

In [ ]:
data_oa_grp_doc_types = dfs.groupby(['publication_year','is_oa']).count().T
data_oa_grp_doc_types['total', 'total'] = data_oa_grp_doc_types.T.sum()
data_oa_grp_doc_types

In [ ]:
doc_types_year = dfs.groupby(['publication_year']).count().T.iloc[1:]
doc_types_year['total'] = doc_types_year.T.sum()
doc_types_year

In [ ]:
for col in data_oa_grp_doc_types.columns:
    for row in data_oa_grp_doc_types.index.to_list():
        data_oa_grp_doc_types[col] = data_oa_grp_doc_types[col]
        data_oa_grp_doc_types[col[0],col[1]+'_%'] = data_oa_grp_doc_types[col]/doc_types_year[col[0]]*100
        data_oa_grp_doc_types[col[0],col[1]+'_%_z_score'] = (data_oa_grp_doc_types[col[0],col[1]+'_%'] - data_oa_grp_doc_types[col[0],col[1]+'_%'].mean()) /data_oa_grp_doc_types[col[0],col[1]+'_%'].std()

#save data
data_oa_grp_doc_types.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_grp_year_doc_types_z_score.csv')
data_oa_grp_doc_types

## group by 'oa_status' 

In [ ]:
data_oa_status_doc_types = data_oa[['publication_year','oa_status','genre']].set_index(['oa_status','publication_year'])
data_oa_status_doc_types

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_status_doc_types[col]) for col in data_oa_status_doc_types]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

In [ ]:
data_oa_status_grp_doc_types = dfs.groupby(['publication_year','oa_status']).count().T
data_oa_status_grp_doc_types['total', 'total'] = data_oa_status_grp_doc_types.T.sum()
data_oa_status_grp_doc_types

In [ ]:
for col in data_oa_status_grp_doc_types.columns:
    for row in data_oa_status_grp_doc_types.index.to_list():
        #data_oa_status_grp_doc_types[col] = data_oa_status_grp_doc_types[col]
        data_oa_status_grp_doc_types[col[0],col[1]+'_%'] = data_oa_status_grp_doc_types[col]/doc_types_year[col[0]]*100
        data_oa_status_grp_doc_types[col[0],col[1]+'_%_z_score'] = (data_oa_status_grp_doc_types[col[0],col[1]+'_%'] - data_oa_status_grp_doc_types[col[0],col[1]+'_%'].mean()) /data_oa_status_grp_doc_types[col[0],col[1]+'_%'].std()

#save data
data_oa_status_grp_doc_types.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_status_grp_year_doc_types_z_score.csv')
data_oa_status_grp_doc_types

# 9. relation between journal and OA ?

In [ ]:
data_oa.columns

## group by 'is_oa' 

In [ ]:
data_oa_journal_name = data_oa[['is_oa','oa_status','journal_name','publication_year']].set_index(['publication_year','oa_status','is_oa'])
data_oa_journal_name

In [ ]:
data_oa_journal_name['journal_name'].value_counts()

In [ ]:
top_100_journal = data_oa_journal_name['journal_name'].value_counts().head(100).to_frame().index.to_list()
top_100_journal

In [ ]:
for journal in top_100_journal:
    data_oa_journal_name[journal] = np.where(data_oa_journal_name['journal_name'] == journal, 1, np.nan) 
    
data_oa_journal_name = data_oa_journal_name.drop('journal_name', axis=1).dropna(how='all').reset_index()
data_oa_journal_name

In [ ]:
data_oa_journal_name.count()

In [ ]:
data_oa_grp_journal_name = data_oa_journal_name.groupby(['publication_year','is_oa']).count().T.iloc[1:]
data_oa_grp_journal_name['total', 'total'] = data_oa_grp_journal_name.T.sum()
data_oa_grp_journal_name

In [ ]:
journal_year = data_oa_journal_name.groupby(['publication_year']).count().T.iloc[2:]
journal_year['total'] = journal_year.T.sum()
journal_year

In [ ]:
for col in data_oa_grp_journal_name.columns:
    for row in data_oa_grp_journal_name.index.to_list():
        data_oa_grp_journal_name[col] = data_oa_grp_journal_name[col]
        data_oa_grp_journal_name[col[0],col[1]+'_%'] = data_oa_grp_journal_name[col]/journal_year[col[0]]*100
        data_oa_grp_journal_name[col[0],col[1]+'_%_z_score'] = (data_oa_grp_journal_name[col[0],col[1]+'_%'] - data_oa_grp_journal_name[col[0],col[1]+'_%'].mean()) /data_oa_grp_journal_name[col[0],col[1]+'_%'].std()

#save data
data_oa_grp_journal_name.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_grp_year_journal_name_z_score_top_100.csv')
data_oa_grp_journal_name

## group by 'oa_status' 

In [ ]:
data_oa_status_grp_journal_name = data_oa_journal_name.groupby(['publication_year','oa_status']).count().T.iloc[1:]
data_oa_status_grp_journal_name['total', 'total'] = data_oa_status_grp_journal_name.T.sum()
data_oa_status_grp_journal_name

In [ ]:
for col in data_oa_status_grp_journal_name.columns:
    for row in data_oa_status_grp_journal_name.index.to_list():
        data_oa_status_grp_journal_name[col] = data_oa_status_grp_journal_name[col]
        data_oa_status_grp_journal_name[col[0],col[1]+'_%'] = data_oa_status_grp_journal_name[col]/journal_year[col[0]]*100
        data_oa_status_grp_journal_name[col[0],col[1]+'_%_z_score'] = (data_oa_status_grp_journal_name[col[0],col[1]+'_%'] - data_oa_status_grp_journal_name[col[0],col[1]+'_%'].mean()) /data_oa_status_grp_journal_name[col[0],col[1]+'_%'].std()

#save data
data_oa_status_grp_journal_name.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_status_grp_year_journal_name_z_score_top100.csv')
data_oa_status_grp_journal_name

# 8. relation between languages and OA ?

In [ ]:
data_oa.columns

In [ ]:
data_oa['Language'].value_counts()

In [ ]:
data_oa_language = data_oa[['publication_year','is_oa','oa_status','Language']].set_index(['publication_year','is_oa','oa_status'])
data_oa_language

In [ ]:
data_oa_language2 = data_oa_language['Language'].str.split('; ', expand=True)
data_oa_language2

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_language2[col]) for col in data_oa_language2]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

## group by 'is_oa' 

In [ ]:
data_oa_language_grp_is_oa = dfs.groupby(['publication_year','is_oa']).count().T.iloc[1:]
data_oa_language_grp_is_oa['total', 'total'] = data_oa_language_grp_is_oa.T.sum()
data_oa_language_grp_is_oa

In [ ]:
language_year = dfs.groupby(['publication_year']).count().T.iloc[2:]
language_year['total'] = language_year.T.sum()
language_year

In [ ]:
for col in data_oa_language_grp_is_oa.columns:
    for row in data_oa_language_grp_is_oa.index.to_list():
        data_oa_language_grp_is_oa[col] = data_oa_language_grp_is_oa[col]
        data_oa_language_grp_is_oa[col[0],col[1]+'_%'] = data_oa_language_grp_is_oa[col]/language_year[col[0]]*100
        data_oa_language_grp_is_oa[col[0],col[1]+'_%_z_score'] = (data_oa_language_grp_is_oa[col[0],col[1]+'_%'] - data_oa_language_grp_is_oa[col[0],col[1]+'_%'].mean()) /data_oa_language_grp_is_oa[col[0],col[1]+'_%'].std()

#save data
data_oa_language_grp_is_oa.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_language_grp_year_is_oa_z_score.csv')
data_oa_language_grp_is_oa

## group by 'oa_status' 

In [ ]:
data_oa_language_grp_oa_status = dfs.groupby(['publication_year','oa_status']).count().T.iloc[1:]
data_oa_language_grp_oa_status['total', 'total'] = data_oa_language_grp_oa_status.T.sum()
data_oa_language_grp_oa_status

In [ ]:
for col in data_oa_language_grp_oa_status.columns:
    for row in data_oa_language_grp_oa_status.index.to_list():
        data_oa_language_grp_oa_status[col] = data_oa_language_grp_oa_status[col]
        data_oa_language_grp_oa_status[col[0],col[1]+'_%'] = data_oa_language_grp_oa_status[col]/language_year[col[0]]*100
        data_oa_language_grp_oa_status[col[0],col[1]+'_%_z_score'] = (data_oa_language_grp_oa_status[col[0],col[1]+'_%'] - data_oa_language_grp_oa_status[col[0],col[1]+'_%'].mean()) /data_oa_language_grp_oa_status[col[0],col[1]+'_%'].std()

#save data
data_oa_language_grp_oa_status.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_language_grp_year_oa_status_z_score.csv')
data_oa_language_grp_oa_status

# 3. What percentage of OA according to research area ?

In [ ]:
data_oa['Research Areas'].value_counts()

In [ ]:
data_oa_research_areas = data_oa[['publication_year','is_oa','oa_status','Research Areas']].set_index(['publication_year','is_oa','oa_status'])
data_oa_research_areas

In [ ]:
data_oa_research_areas2 = data_oa_research_areas['Research Areas'].str.split('; ', expand=True)
data_oa_research_areas2

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_research_areas2[col]) for col in data_oa_research_areas2]
d = reduce(lambda x,y : x.add(y, fill_value=0), dfs)

# data cleaning
cols = [c for c in d.columns if c[-1:] != '.']
df=d[cols]
cols = [c for c in df.columns.to_list() if len(c) != 1]
df=df[cols]

dfs= df.replace(0.0, np.nan).reset_index()
dfs

## group by 'is_oa' 

In [ ]:
data_oa_research_areas_grp_is_oa = dfs.groupby(['publication_year','is_oa']).count().T.iloc[1:]
data_oa_research_areas_grp_is_oa['total', 'total'] = data_oa_research_areas_grp_is_oa.T.sum()
data_oa_research_areas_grp_is_oa

In [ ]:
research_area_year = dfs.groupby(['publication_year']).count().T.iloc[2:]
research_area_year['total'] = research_area_year.T.sum()
research_area_year

In [ ]:
for col in data_oa_research_areas_grp_is_oa.columns:
    for row in data_oa_research_areas_grp_is_oa.index.to_list():
        data_oa_research_areas_grp_is_oa[col[0],col[1]+'_%'] = data_oa_research_areas_grp_is_oa[col]/research_area_year[col[0]]*100
        data_oa_research_areas_grp_is_oa[col[0],col[1]+'_%_z_score'] = (data_oa_research_areas_grp_is_oa[col[0],col[1]+'_%'] - data_oa_research_areas_grp_is_oa[col[0],col[1]+'_%'].mean()) /data_oa_research_areas_grp_is_oa[col[0],col[1]+'_%'].std()

#save data
data_oa_research_areas_grp_is_oa.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_research_areas_grp_year_is_oa_z_score.csv')
data_oa_research_areas_grp_is_oa

## group by 'oa_status' 

In [ ]:
data_oa_research_areas_grp_oa_status = dfs.groupby(['publication_year','oa_status']).count().T.iloc[1:]
data_oa_research_areas_grp_oa_status['total', 'total'] = data_oa_research_areas_grp_oa_status.T.sum()
data_oa_research_areas_grp_oa_status

In [ ]:
for col in data_oa_research_areas_grp_oa_status.columns:
    for row in data_oa_research_areas_grp_oa_status.index.to_list():
        data_oa_research_areas_grp_oa_status[col[0],col[1]+'_%'] = data_oa_research_areas_grp_oa_status[col]/research_area_year[col[0]]*100
        data_oa_research_areas_grp_oa_status[col[0],col[1]+'_%_z_score'] = (data_oa_research_areas_grp_oa_status[col[0],col[1]+'_%'] - data_oa_research_areas_grp_oa_status[col[0],col[1]+'_%'].mean()) /data_oa_research_areas_grp_oa_status[col[0],col[1]+'_%'].std()

#save data
data_oa_research_areas_grp_oa_status.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_research_areas_grp_year_oa_status_z_score.csv')
data_oa_research_areas_grp_oa_status

In [ ]:
xxxxxxxxx

# 4. are OA are highly cited ? 

- scholarly impact of OA
- what types of OA increased impact

In [ ]:
data_oa.columns

In [ ]:
data_oa['times_cited_group'].value_counts().to_frame().tail(60)

In [ ]:
data_oa_times_cited_group = data_oa[['wos_ID','is_oa','oa_status','times_cited_group']].set_index(['wos_ID','is_oa','oa_status'])
data_oa_times_cited_group

In [ ]:
# data_oa_research_areas2 = data_oa_research_areas['Research Areas'].str.split('; ', expand=True)
# data_oa_research_areas2

In [ ]:
# get dummies to have contries on columns
from functools import reduce
dfs = [pd.get_dummies(data_oa_times_cited_group[col]) for col in data_oa_times_cited_group]
dfs = reduce(lambda x,y : x.add(y, fill_value=0), dfs).reset_index().replace({0:np.nan})
dfs

## group by 'is_oa' 

In [ ]:
data_oa_times_cited_group_grp_is_oa = dfs.groupby(['is_oa']).count()
data_oa_grp_funding['total', 'total'] = data_oa_grp_funding.T.sum()
data_oa_times_cited_group_grp_is_oa

In [ ]:
data_oa_times_cited_group_count = dfs.count().to_frame().T
data_oa_times_cited_group_count

In [ ]:
for col in data_oa_times_cited_group_grp_is_oa.columns:
    data_oa_times_cited_group_grp_is_oa[col[0],col[1]+'_%'] = data_oa_times_cited_group_grp_is_oa[col]/data_oa_times_cited_group_count[col][0]*100;
        

data_oa_times_cited_group_grp_is_oa_T = data_oa_times_cited_group_grp_is_oa.T.sort_index(ascending=False)
#save data
data_oa_times_cited_group_grp_is_oa_T.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_group_grp_is_oa.csv')
data_oa_times_cited_group_grp_is_oa_T

## group by 'oa_status' 

In [ ]:
data_oa_times_cited_group_oa_status = dfs.groupby(['oa_status']).count()
data_oa_grp_funding['total', 'total'] = data_oa_grp_funding.T.sum()
data_oa_times_cited_group_oa_status

In [ ]:
for col in data_oa_times_cited_group_oa_status.columns:
    data_oa_times_cited_group_oa_status[col[0],col[1]+'_%'] = data_oa_times_cited_group_oa_status[col]/data_oa_times_cited_group_count[col][0]*100;
        

data_oa_times_cited_group_oa_status_T = data_oa_times_cited_group_oa_status.T.sort_index(ascending=False)
#save data
data_oa_times_cited_group_oa_status_T.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_group_oa_status.csv')
data_oa_times_cited_group_oa_status_T

# 4 b. impact of 'Times Cited, All Databases' on OA

In [ ]:
data_oa_times_cited = data_oa[['wos_ID','is_oa','publication_year','Times Cited, All Databases']].set_index(['wos_ID'])
data_oa_times_cited

## group by 'is_oa' 

In [ ]:
data_oa_times_cited_is_oa_mean = data_oa_times_cited.groupby(['is_oa']).mean().rename(columns={'Times Cited, All Databases':'times_cited_mean'})
data_oa_times_cited_is_oa_mean['times_cited_sum'] = data_oa_times_cited.groupby(['is_oa']).sum()
data_oa_times_cited_is_oa_mean['times_cited_std'] = data_oa_times_cited.groupby(['is_oa']).std()
data_oa_times_cited_is_oa_mean['times_cited_median'] = data_oa_times_cited.groupby(['is_oa']).median()
# data_oa_times_cited_is_oa_mean['times_cited_max'] = data_oa_times_cited.groupby(['is_oa']).max()
# data_oa_times_cited_is_oa_mean['times_cited_min'] = data_oa_times_cited.groupby(['is_oa']).min()

data_oa_times_cited_is_oa_mean_sum_std_median = data_oa_times_cited_is_oa_mean.reset_index()

#save data
data_oa_times_cited_is_oa_mean_sum_std_median.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_is_oa_mean_sum_std_median.csv')
data_oa_times_cited_is_oa_mean_sum_std_median

In [ ]:
data_oa_times_cited_is_oa_year_mean = data_oa_times_cited.groupby(['publication_year','is_oa']).mean().rename(columns={'Times Cited, All Databases':'times_cited_mean'})
data_oa_times_cited_is_oa_year_mean['times_cited_sum'] = data_oa_times_cited.groupby(['publication_year','is_oa']).sum()
data_oa_times_cited_is_oa_year_mean['times_cited_std'] = data_oa_times_cited.groupby(['publication_year','is_oa']).std()
data_oa_times_cited_is_oa_year_mean['times_cited_median'] = data_oa_times_cited.groupby(['publication_year','is_oa']).median()

data_oa_times_cited_is_oa_year_mean_sum_std_median = data_oa_times_cited_is_oa_year_mean.reset_index()
#save data
data_oa_times_cited_is_oa_year_mean_sum_std_median.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_is_oa_year_mean_sum_std_median.csv')
data_oa_times_cited_is_oa_year_mean_sum_std_median

## group by 'oa_status' 

In [ ]:
data_oa_times_cited_oa_status = data_oa[['wos_ID','oa_status','publication_year','Times Cited, All Databases']].set_index(['wos_ID'])
data_oa_times_cited_oa_status

In [ ]:
data_oa_times_cited_oa_status_mean = data_oa_times_cited_oa_status.groupby(['oa_status']).mean().rename(columns={'Times Cited, All Databases':'times_cited_mean'})
data_oa_times_cited_oa_status_mean['times_cited_sum'] = data_oa_times_cited_oa_status.groupby(['oa_status']).sum()
data_oa_times_cited_oa_status_mean['times_cited_std'] = data_oa_times_cited_oa_status.groupby(['oa_status']).std()
data_oa_times_cited_oa_status_mean['times_cited_median'] = data_oa_times_cited_oa_status.groupby(['oa_status']).median()
# data_oa_times_cited_is_oa_mean['times_cited_max'] = data_oa_times_cited.groupby(['is_oa']).max()
# data_oa_times_cited_is_oa_mean['times_cited_min'] = data_oa_times_cited.groupby(['is_oa']).min()

data_oa_times_cited_oa_status_mean_sum_std_median = data_oa_times_cited_oa_status_mean.reset_index()

#save data
data_oa_times_cited_oa_status_mean_sum_std_median.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_oa_status_mean_sum_std_median.csv')
data_oa_times_cited_oa_status_mean_sum_std_median

In [ ]:
data_oa_times_cited_oa_status_year_mean = data_oa_times_cited_oa_status.groupby(['publication_year','oa_status']).mean().rename(columns={'Times Cited, All Databases':'times_cited_mean'})
data_oa_times_cited_oa_status_year_mean['times_cited_sum'] = data_oa_times_cited_oa_status.groupby(['publication_year','oa_status']).sum()
data_oa_times_cited_oa_status_year_mean['times_cited_std'] = data_oa_times_cited_oa_status.groupby(['publication_year','oa_status']).std()
data_oa_times_cited_oa_status_year_mean['times_cited_median'] = data_oa_times_cited_oa_status.groupby(['publication_year','oa_status']).median()

data_oa_times_cited_oa_status_year_mean_sum_std_median = data_oa_times_cited_oa_status_year_mean.reset_index()
#save data
data_oa_times_cited_oa_status_year_mean_sum_std_median.to_csv('D:\RESEARCH IN AFRICA\Manuscripts/Open access paper/data_analysis/tables/data_oa_times_cited_oa_status_year_mean_sum_std_median.csv')
data_oa_times_cited_oa_status_year_mean_sum_std_median